# Estimation fit: model vs data moments

Load estimated parameters from a completed CE run, solve + simulate at those values, and compare simulated moments against empirical targets from `moments_data.csv`.

Set `RESULTS_DIR` below to point at a finished estimation run (contains `theta_best.json`, `summary.json`, `fit_table.csv`, `convergence.csv`).

In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*IProgress.*')

import json, os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

REPO_ROOT = os.path.abspath('../../..')
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)

# ── Configuration ──
# Results are saved in two places:
#   1. Remote:  /g/data/tp66/results/durables2_0/estimation/<spec>/<run>/
#   2. Local:   mod/<syntax>/estimation/results/<spec>/<run>/
#
# Point RESULTS_DIR at whichever copy is accessible.
# List available local runs:
MOD_DIR = 'examples/durables2_0/mod/separable'
local_results_root = os.path.join(MOD_DIR, 'estimation', 'results')
if os.path.isdir(local_results_root):
    for spec in sorted(os.listdir(local_results_root)):
        spec_dir = os.path.join(local_results_root, spec)
        if os.path.isdir(spec_dir):
            runs = sorted(os.listdir(spec_dir))
            for r in runs:
                print(f'  {os.path.join(spec_dir, r)}')
else:
    print('No local results yet.')

# Set this to a specific run:
RESULTS_DIR = os.path.join(local_results_root, 'baseline', 'est_YYYYMMDD_HHMMSS')

# Solver settings for re-simulation
GRID = 300
N_SIM = 10_000
CALIB_OVERRIDES = {'t0': 20}
SETTING_OVERRIDES = {'n_a': GRID, 'n_h': GRID, 'n_w': GRID}

print(f'\nResults: {RESULTS_DIR}')
print(f'Mod:     {MOD_DIR}')

## 1. Load estimation results

In [ ]:
# Load theta_best and summary from the results directory
with open(os.path.join(RESULTS_DIR, 'theta_best.json')) as f:
    theta_best = json.load(f)

with open(os.path.join(RESULTS_DIR, 'summary.json')) as f:
    summary = json.load(f)

theta_mean = summary.get('theta_mean', theta_best)
theta_se = summary.get('theta_se', {})

# Display parameter estimates
print(f"Loss:      {summary['objective']:.6f}")
print(f"Converged: {summary['converged']} ({summary['n_iter']} iters)")
print()
print(f"{'param':12s} {'best':>10s} {'mean':>10s} {'SE':>10s}")
print('-' * 44)
for p in sorted(theta_best):
    se = theta_se.get(p, float('nan'))
    mn = theta_mean.get(p, float('nan')) if isinstance(theta_mean, dict) else float('nan')
    print(f"{p:12s} {theta_best[p]:10.4f} {mn:10.4f} {se:10.6f}")

## 2. Convergence

In [ ]:
# Convergence plot from saved CSV
conv_path = os.path.join(RESULTS_DIR, 'convergence.csv')
conv = pd.read_csv(conv_path)

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(conv['iter'], conv['best_loss'], 'o-', label='Best loss', markersize=4)
ax.plot(conv['iter'], conv['elite_mean_loss'], 's--', label='Elite mean loss',
        markersize=4, alpha=0.7)
ax.set_xlabel('CE iteration')
ax.set_ylabel('SMM loss')
ax.set_yscale('log')
ax.legend()
ax.set_title('Cross-entropy convergence')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 3. Solve at estimated parameters and simulate

In [ ]:
from examples.durables2_0.solve import solve
from examples.durables2_0.horses.simulate import simulate_lifecycle

# Merge estimated params into calibration overrides
calib = {**CALIB_OVERRIDES, **theta_best}
print(f'Solving at theta_best: {theta_best}')

nest, grids = solve(
    MOD_DIR,
    calib_overrides=calib,
    setting_overrides=SETTING_OVERRIDES,
    verbose='summary',
)

sim_data = simulate_lifecycle(nest, grids, N=N_SIM, seed=99)
print(f'Simulation: {N_SIM} agents, {sim_data["c"].shape[0]} periods')

## 4. Compute model moments and load data moments

In [ ]:
from pathlib import Path
from kikku.run.estimate import load_estimation_spec
from kikku.run.moments import make_moment_fn

# Load estimation spec (to get moment_spec and data_moments from CSV)
spec_path = os.path.join(MOD_DIR, 'estimation', 'baseline.yaml')
spec = load_estimation_spec(spec_path)
moment_spec = spec['moment_spec']
data_moments = spec['data_moments']

# Compute model moments from simulation
moment_fn = make_moment_fn(moment_spec)
sim_moments = moment_fn(sim_data)

# Filter to target keys (those in the CSV) -- ignore bulk identification keys
target_keys = sorted(k for k in data_moments if k in sim_moments)
print(f'Data moments:  {len(data_moments)} keys')
print(f'Sim moments:   {len(sim_moments)} keys')
print(f'Matched keys:  {len(target_keys)}')

## 5. Fit table

In [ ]:
# Build fit table: data vs model for each targeted moment
import re

rows = []
for k in target_keys:
    d = data_moments[k]
    s = sim_moments.get(k, float('nan'))
    if np.isnan(d):
        continue
    resid = s - d
    denom = d**2 if abs(d) >= 1.0 else 1.0
    contrib = resid**2 / max(denom, 1e-20)
    rows.append({'moment': k, 'data': d, 'model': s,
                 'residual': resid, 'contribution': contrib})

fit_df = pd.DataFrame(rows)
fit_df['pct_dev'] = 100 * fit_df['residual'] / fit_df['data'].where(fit_df['data'].abs() >= 1, 1.0)
fit_df = fit_df.sort_values('contribution', ascending=False)

total_loss = fit_df['contribution'].sum()
fit_df['share'] = 100 * fit_df['contribution'] / total_loss

print(f'Total SMM loss: {total_loss:.4f}')
print()
fit_df[['moment', 'data', 'model', 'pct_dev', 'share']].to_string(
    index=False, float_format='%.3f', formatters={
        'moment': '{:40s}'.format, 'share': '{:6.1f}%'.format})

## 6. Model vs data by age group

Targeted moments plotted by age group: means, standard deviations, correlations, autocorrelations.

In [ ]:
# Parse moment keys into (base_name, age_group) and group by base_name
def parse_key(k):
    m = re.match(r'^(.+)__age(\d+)$', k)
    if m:
        return m.group(1), int(m.group(2))
    return k, None

# Group targeted moments by base name
from collections import defaultdict
moment_groups = defaultdict(list)
for k in target_keys:
    base, ag = parse_key(k)
    d = data_moments.get(k, float('nan'))
    s = sim_moments.get(k, float('nan'))
    if not np.isnan(d):
        moment_groups[base].append((ag, d, s))

# Age group midpoints for x-axis
age_mid = {1: 22, 2: 27, 3: 32, 4: 37, 5: 42, 6: 47, 7: 52, 8: 57, 9: 62}

# Friendly labels
LABELS = {
    'av_consumption2_14_0': 'Mean consumption',
    'av_wealth_real_14_0': 'Mean housing (wealth_real)',
    'av_fin_assets_14_0': 'Mean financial assets',
    'sd_consumption2_14_0': 'SD consumption',
    'sd_wealth_real_14_0': 'SD housing',
    'sd_fin_assets_14_0': 'SD financial assets',
    'corr_cons_realw_14_0': 'Corr(c, housing)',
    'autocorr_wealth_real_0': 'Autocorr housing',
    'autocorr_wealth_fin_0': 'Autocorr financial',
    'autocorr_consumption_0': 'Autocorr consumption',
}

# --- Means ---
mean_bases = [b for b in moment_groups if b.startswith('av_')]
n_mean = len(mean_bases)
if n_mean:
    fig, axes = plt.subplots(1, n_mean, figsize=(5 * n_mean, 4), squeeze=False)
    for i, base in enumerate(sorted(mean_bases)):
        ax = axes[0, i]
        pts = sorted(moment_groups[base])
        ages = [age_mid.get(ag, ag) for ag, _, _ in pts]
        data_vals = [d for _, d, _ in pts]
        model_vals = [s for _, _, s in pts]
        ax.plot(ages, data_vals, 'ko-', label='Data', markersize=6)
        ax.plot(ages, model_vals, 'rs--', label='Model', markersize=6)
        ax.set_xlabel('Age')
        ax.set_title(LABELS.get(base, base))
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle('Means by age group', fontsize=13)
    fig.tight_layout()
    plt.show()

# --- Standard deviations ---
sd_bases = [b for b in moment_groups if b.startswith('sd_')]
n_sd = len(sd_bases)
if n_sd:
    fig, axes = plt.subplots(1, n_sd, figsize=(5 * n_sd, 4), squeeze=False)
    for i, base in enumerate(sorted(sd_bases)):
        ax = axes[0, i]
        pts = sorted(moment_groups[base])
        ages = [age_mid.get(ag, ag) for ag, _, _ in pts]
        data_vals = [d for _, d, _ in pts]
        model_vals = [s for _, _, s in pts]
        ax.plot(ages, data_vals, 'ko-', label='Data', markersize=6)
        ax.plot(ages, model_vals, 'rs--', label='Model', markersize=6)
        ax.set_xlabel('Age')
        ax.set_title(LABELS.get(base, base))
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
    fig.suptitle('Standard deviations by age group', fontsize=13)
    fig.tight_layout()
    plt.show()

# --- Correlations and autocorrelations ---
other_bases = [b for b in moment_groups if not b.startswith('av_') and not b.startswith('sd_')]
n_other = len(other_bases)
if n_other:
    fig, axes = plt.subplots(1, n_other, figsize=(4.5 * n_other, 4), squeeze=False)
    for i, base in enumerate(sorted(other_bases)):
        ax = axes[0, i]
        pts = sorted(moment_groups[base])
        ages = [age_mid.get(ag, ag) for ag, _, _ in pts]
        data_vals = [d for _, d, _ in pts]
        model_vals = [s for _, _, s in pts]
        ax.plot(ages, data_vals, 'ko-', label='Data', markersize=6)
        ax.plot(ages, model_vals, 'rs--', label='Model', markersize=6)
        ax.set_xlabel('Age')
        ax.set_title(LABELS.get(base, base))
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(-1, 1)
    fig.suptitle('Correlations / autocorrelations by age group', fontsize=13)
    fig.tight_layout()
    plt.show()

## 7. Lifecycle profiles at estimated parameters

In [ ]:
# Lifecycle: mean consumption, assets, housing by age
stage0 = nest['periods'][0]['stages']['keeper_cons']
t0 = int(stage0.calibration['t0'])
T = int(stage0.settings['T'])

ages = np.arange(t0, T)
mean_c = np.nanmean(sim_data['c'][t0:T], axis=1)
mean_a = np.nanmean(sim_data['a'][t0:T], axis=1)
mean_h = np.nanmean(sim_data['h'][t0:T], axis=1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ages, mean_c, 'b-', linewidth=2)
axes[0].set_title('Mean consumption')
axes[0].set_xlabel('Age')
axes[0].grid(True, alpha=0.3)

axes[1].plot(ages, mean_a, 'g-', linewidth=2)
axes[1].set_title('Mean financial assets')
axes[1].set_xlabel('Age')
axes[1].grid(True, alpha=0.3)

axes[2].plot(ages, mean_h, 'r-', linewidth=2)
axes[2].set_title('Mean housing')
axes[2].set_xlabel('Age')
axes[2].grid(True, alpha=0.3)

fig.suptitle(f'Lifecycle profiles at estimated parameters (N={N_SIM})', fontsize=13)
fig.tight_layout()
plt.show()

# Adjustment rate by age
if 'discrete' in sim_data:
    d = sim_data['discrete'][t0:T]
    adj_rate = np.nanmean(d, axis=1)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.plot(ages, adj_rate * 100, 'k-', linewidth=2)
    ax.set_xlabel('Age')
    ax.set_ylabel('Adjustment rate (%)')
    ax.set_title('Housing adjustment rate by age')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

## 8. Loss decomposition

Bar chart showing which moments contribute most to the total SMM loss.

In [ ]:
# Top contributors to loss
top = fit_df.head(15).copy()
top['label'] = top['moment'].apply(
    lambda k: LABELS.get(parse_key(k)[0], parse_key(k)[0]) + f' (age {parse_key(k)[1]})')

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top)), top['share'].values, color='steelblue')
ax.set_yticks(range(len(top)))
ax.set_yticklabels(top['label'].values, fontsize=9)
ax.set_xlabel('Share of total loss (%)')
ax.set_title('Top 15 moment contributions to SMM loss')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
fig.tight_layout()
plt.show()